# XBRL Tag Quality Review

**Prepared for:** Apple Inc. management  
**Prepared by:** Outside XBRL reporting consultant  
**Benchmark company:** Microsoft Corporation  
**Purpose:** Assess standard-tag usage and identify Apple extension concepts that may be representable with an existing standard tag.

This review uses the SEC Financial Statement Data Sets and compares tags actually reported in each company's latest 10-K or 10-Q in the selected quarterly dataset. The analysis is a screening tool: a high text-similarity score is a candidate for accounting review, not proof that two concepts are equivalent.

## Executive recommendation and scope

Prefer an available US GAAP standard concept when it faithfully represents the reported accounting meaning, including its scope, period type, unit, and dimensional context. Create or retain an extension only when no standard concept is appropriate; an extension can be justified for a genuinely company-specific disclosure. Replacing an extension solely to reduce its count can make a filing less accurate.

Apple is benchmarked against Microsoft as a large public technology company. The comparison is directional, not a peer-quality ranking: differences in products, accounting policies, filing scope, and disclosure detail affect the tag mix. The final recommendations below are generated from the selected filings and provide specific candidates for a human review.

## 1. Load the SEC filing tables

The notebook follows the SEC-table loading and filing-selection approach in the project's `Liam_XBRL_FS.ipynb` and `xbrl_fs.create_financial_statements` helper. It reads `SUB`, `NUM`, `TAG`, and `PRE` from one SEC ZIP, then joins the selected filing's reported facts to its tag definitions. The configured archive is the 2026 Q2 dataset, the latest posted quarter as of October 15, 2026. Set `SEC_USER_AGENT` in the environment to a real name and contact email before downloading, as requested by the SEC.

In [1]:
from __future__ import annotations

import os
import re
import sys
import urllib.request
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
from IPython.display import Markdown, display
from rapidfuzz import fuzz, process

PROJECT_ROOT = next(
    (parent for parent in Path.cwd().resolve().parents if (parent / "xbrl_fs").is_dir()),
    Path.cwd().resolve() if (Path.cwd().resolve() / "xbrl_fs").is_dir() else None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing the xbrl_fs package.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from xbrl_fs import create_financial_statements

DATASET_URL = "https://www.sec.gov/files/dera/data/financial-statement-data-sets/2026q2.zip"
DATASET_FILE = Path(os.environ.get("SEC_DATASET_FILE", PROJECT_ROOT / "2026q2.zip"))
SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "").strip()

if not DATASET_FILE.exists():
    if not SEC_USER_AGENT or "@" not in SEC_USER_AGENT:
        raise ValueError(
            "Set SEC_USER_AGENT to 'Your Name your.email@example.com' before downloading SEC data."
        )
    request = urllib.request.Request(DATASET_URL, headers={"User-Agent": SEC_USER_AGENT})
    with urllib.request.urlopen(request, timeout=120) as response:
        DATASET_FILE.write_bytes(response.read())

print("SEC dataset ready.")

SEC dataset ready.


In [2]:
with ZipFile(DATASET_FILE) as archive:
    members = {Path(name).name.lower(): name for name in archive.namelist()}
    sec_tables = {}
    for table_name in ("sub", "num", "tag", "pre"):
        member = next(
            (members.get(f"{table_name}.{extension}") for extension in ("txt", "tsv")
             if members.get(f"{table_name}.{extension}") is not None),
            None,
        )
        if member is None:
            raise ValueError(f"SEC ZIP is missing the {table_name.upper()} table.")
        with archive.open(member) as file:
            sec_tables[table_name] = pd.read_csv(file, sep="\t", dtype=str, low_memory=False)

submissions = sec_tables["sub"]
numeric = sec_tables["num"]
tags = sec_tables["tag"]
presentation = sec_tables["pre"]
for frame in (submissions, numeric, tags, presentation):
    frame.columns = [str(column).lower() for column in frame.columns]

required_tag_columns = {"tag", "version", "custom", "tlabel"}
missing_tag_columns = required_tag_columns.difference(tags.columns)
if missing_tag_columns:
    raise ValueError(f"TAG table is missing required columns: {sorted(missing_tag_columns)}")

company_ciks = {"Apple Inc.": "320193", "Microsoft Corporation": "789019"}
filings = {}
financial_statements = {}
for company_name, cik in company_ciks.items():
    filing, statements = create_financial_statements(
        submissions, numeric, tags, cik, presentation=presentation
    )
    filings[company_name] = filing
    financial_statements[company_name] = statements

filing_summary = pd.DataFrame(
    [
        {
            "Company": company_name,
            "CIK": f"{int(company_ciks[company_name]):010d}",
            "Form": filing["form"],
            "Filed": filing["filed"],
            "Period ended": filing.get("period", "not reported"),
            "Accession": filing["adsh"],
        }
        for company_name, filing in filings.items()
    ]
)
display(filing_summary)


,Company,CIK,Form,Filed,Period ended,Accession
0,Apple Inc.,0000320193,10-Q,2026-05-01,20260331,0000320193-26-000013
1,Microsoft Corporation,0000789019,10-Q,2026-04-29,20260331,0001193125-26-191507


## 2. Identify the standard and extension tags actually used

For each selected accession, `NUM` supplies the concepts used in the filing and `TAG` supplies their `custom` classification and human-readable labels. The two sets below are explicitly filtered on `TAG.custom` and deduplicated with `.unique()`. The tables display every distinct tag name, not just a sample.

In [3]:
def filing_tag_metadata(filing: pd.Series) -> pd.DataFrame:
    """Return distinct TAG metadata for concepts reported in this accession."""
    filing_facts = numeric.loc[numeric["adsh"].eq(filing["adsh"])]
    fact_keys = ["tag", "version"]
    missing = set(fact_keys).difference(filing_facts.columns)
    if missing:
        raise ValueError(f"NUM table is missing required tag keys: {sorted(missing)}")
    used_keys = filing_facts[fact_keys].drop_duplicates()
    tag_definitions = tags.drop_duplicates(fact_keys, keep="last")
    used = used_keys.merge(
        tag_definitions,
        on=fact_keys,
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    unmatched = used.loc[used["_merge"].ne("both"), fact_keys]
    if not unmatched.empty:
        raise ValueError(f"Some filing concepts have no matching TAG metadata:\n{unmatched.head(10)}")
    used["custom"] = used["custom"].astype(str).str.strip()
    unexpected = sorted(set(used["custom"].dropna()) - {"0", "1"})
    if unexpected:
        raise ValueError(f"Unexpected TAG.custom values: {unexpected}")
    return used.drop(columns="_merge").drop_duplicates(["tag", "version"])


used_tags = {
    company_name: filing_tag_metadata(filing)
    for company_name, filing in filings.items()
}
tag_names = {}
for company_name, used in used_tags.items():
    standard_names = used.loc[used["custom"].eq("0"), "tag"].unique()
    extension_names = used.loc[used["custom"].eq("1"), "tag"].unique()
    tag_names[company_name] = {
        "standard": sorted(standard_names.tolist()),
        "extension": sorted(extension_names.tolist()),
    }
    print(
        f"{company_name}: {len(standard_names):,} distinct standard tags; "
        f"{len(extension_names):,} distinct extension tags"
    )
    print("Standard tags (TAG.custom == 0):")
    display(pd.DataFrame({"Standard tag": tag_names[company_name]["standard"]}))
    print("Extension tags (TAG.custom == 1):")
    display(pd.DataFrame({"Extension tag": tag_names[company_name]["extension"]}))


Apple Inc.: 88 distinct standard tags; 1 distinct extension tags
Standard tags (TAG.custom == 0):


,Standard tag
0,AccountsPayableCurrent
1,AccountsReceivableNetCurrent
2,AccumulatedOtherComprehensiveIncomeLossNetOfTax
3,AdjustmentsRelatedToTaxWithholdingForShareBase...
4,AdjustmentsToAdditionalPaidInCapitalSharebased...
...,...
83,StockIssuedDuringPeriodValueNewIssues
84,StockRepurchasedAndRetiredDuringPeriodValue
85,StockholdersEquity
86,WeightedAverageNumberOfDilutedSharesOutstanding


Extension tags (TAG.custom == 1):


,Extension tag
0,IntangibleAssetsNetExcludingGoodwillNoncurrent


Microsoft Corporation: 92 distinct standard tags; 4 distinct extension tags
Standard tags (TAG.custom == 0):


,Standard tag
0,AccountsPayableCurrent
1,AccountsReceivableNetCurrent
2,AccruedIncomeTaxesCurrent
3,AccruedIncomeTaxesNoncurrent
4,AccumulatedDepreciationDepletionAndAmortizatio...
...,...
87,StockRepurchasedAndRetiredDuringPeriodValue
88,StockholdersEquity
89,StockholdersEquityOther
90,WeightedAverageNumberOfDilutedSharesOutstanding


Extension tags (TAG.custom == 1):


,Extension tag
0,AcquisitionsNetOfCashAcquiredAndPurchasesOfInt...
1,DepreciationAmortizationAndOther
2,GainLossOnInvestmentsAndDerivativeInstruments
3,ProceedsFromInvestments


In [4]:
tag_mix_rows = []
for company_name, names in tag_names.items():
    standard_count = len(names["standard"])
    extension_count = len(names["extension"])
    total = standard_count + extension_count
    tag_mix_rows.append(
        {
            "Company": company_name,
            "Standard tags": standard_count,
            "Extension tags": extension_count,
            "Distinct tags": total,
            "Extension share": extension_count / total if total else 0.0,
        }
    )
tag_mix = pd.DataFrame(tag_mix_rows).set_index("Company")
tag_mix_display = tag_mix.copy()
tag_mix_display["Extension share"] = tag_mix_display["Extension share"].map("{:.1%}".format)
display(tag_mix_display)

,Standard tags,Extension tags,Distinct tags,Extension share
Company,,,,
Apple Inc.,88,1,89,1.1%
Microsoft Corporation,92,4,96,4.2%


## 3. Fuzzy matching and preprocessing edge cases

RapidFuzz's `WRatio` is applied consistently while preprocessing varies one meaningful edge case at a time: preserve original case, lowercase without splitting camel case, split camel case and lowercase, split/lowercase and sort tokens, or split/lowercase and normalize punctuation. The comparison reports each extension's best score against Apple's standard concepts and aggregate score coverage. Token sorting is tested as a sensitivity case, not assumed to be superior: XBRL word order can encode important distinctions such as gross/net or current/noncurrent.

The selected method is camel-case splitting plus case-folding while preserving token order. It is robust to capitalization and makes concatenated XBRL names readable, but avoids treating differently ordered accounting concepts as identical. Scores prioritize candidates for review; they do not establish accounting equivalence.

In [5]:
def split_camel_case(value: str) -> str:
    value = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", value)
    return re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", value)


def preserve_case(value: str) -> str:
    return value


def lowercase_raw(value: str) -> str:
    return value.casefold()


def camel_case_lower(value: str) -> str:
    return split_camel_case(value).casefold()


def camel_case_lower_sorted(value: str) -> str:
    return " ".join(sorted(camel_case_lower(value).split()))


def camel_case_alphanumeric(value: str) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", camel_case_lower(value)))


preprocessing_methods = {
    "Original case, raw name": preserve_case,
    "Lowercase, raw name": lowercase_raw,
    "Camel split + lowercase (selected)": camel_case_lower,
    "Camel split + lowercase + sorted words": camel_case_lower_sorted,
    "Camel split + lowercase + alphanumeric normalization": camel_case_alphanumeric,
}

def best_matches(extension_names: list[str], standard_names: list[str], processor) -> pd.DataFrame:
    if not standard_names:
        return pd.DataFrame(columns=["Extension tag", "Candidate standard tag", "Score"])
    rows = []
    for extension in extension_names:
        result = process.extractOne(
            extension,
            standard_names,
            scorer=fuzz.WRatio,
            processor=processor,
        )
        if result is None:
            rows.append({"Extension tag": extension, "Candidate standard tag": None, "Score": 0.0})
        else:
            candidate, score, _ = result
            rows.append(
                {"Extension tag": extension, "Candidate standard tag": candidate, "Score": score}
            )
    return pd.DataFrame(rows)


apple_extensions = tag_names["Apple Inc."]["extension"]
apple_standard = tag_names["Apple Inc."]["standard"]
edge_case_rows = []
for method_name, processor in preprocessing_methods.items():
    matched = best_matches(apple_extensions, apple_standard, processor)
    edge_case_rows.append(
        {
            "Preprocessing edge case": method_name,
            "Median best score": matched["Score"].median() if not matched.empty else float("nan"),
            "Mean best score": matched["Score"].mean() if not matched.empty else float("nan"),
            "Extensions scoring >= 85": int(matched["Score"].ge(85).sum()),
            "Extensions evaluated": len(matched),
        }
    )
edge_case_results = pd.DataFrame(edge_case_rows)
display(edge_case_results.round(1))

,Preprocessing edge case,Median best score,Mean best score,Extensions scoring >= 85,Extensions evaluated
0,"Original case, raw name",90.0,90.0,1,1
1,"Lowercase, raw name",90.0,90.0,1,1
2,Camel split + lowercase (selected),85.5,85.5,1,1
3,Camel split + lowercase + sorted words,85.5,85.5,1,1
4,Camel split + lowercase + alphanumeric normali...,85.5,85.5,1,1


## 4. Compare Apple extensions with standard tags at both companies

For each Apple extension, the chosen `WRatio` method finds the closest standard tag used by Apple and by Microsoft in their selected filings. A candidate that appears in both standard-tag sets is especially useful for comparability, but the companies' definitions and reported facts still need review. Similarity bands below are triage labels only: **strong candidate** (90+), **review candidate** (80–<90), and **weak text match** (<80).

In [6]:
chosen_processor = camel_case_lower
microsoft_standard = tag_names["Microsoft Corporation"]["standard"]
apple_to_apple = best_matches(apple_extensions, apple_standard, chosen_processor)
apple_to_microsoft = best_matches(apple_extensions, microsoft_standard, chosen_processor)

candidate_comparison = apple_to_apple.rename(
    columns={"Candidate standard tag": "Apple standard candidate", "Score": "Apple score"}
).merge(
    apple_to_microsoft.rename(
        columns={
            "Candidate standard tag": "Microsoft standard candidate",
            "Score": "Microsoft score",
        }
    ),
    on="Extension tag",
    how="outer",
    validate="one_to_one",
)
shared_standard_tags = set(apple_standard).intersection(microsoft_standard)
candidate_comparison["Microsoft candidate used by Apple"] = candidate_comparison[
    "Microsoft standard candidate"
].isin(apple_standard)
candidate_comparison["Candidate used by both companies"] = candidate_comparison[
    "Apple standard candidate"
].isin(shared_standard_tags)
candidate_comparison["Review tier"] = candidate_comparison["Apple score"].map(
    lambda score: "Strong candidate" if score >= 90 else "Review candidate" if score >= 80 else "Weak text match"
)
candidate_comparison = candidate_comparison.sort_values(
    ["Apple score", "Microsoft score"], ascending=False
).reset_index(drop=True)
display(candidate_comparison.round({"Apple score": 1, "Microsoft score": 1}))

strong_candidates = candidate_comparison.loc[
    candidate_comparison["Apple score"].ge(90)
].copy()
print(
    f"Strong text candidates: {len(strong_candidates):,} of "
    f"{len(apple_extensions):,} Apple extensions. Each requires definition and context review."
)

,Extension tag,Apple standard candidate,Apple score,Microsoft standard candidate,Microsoft score,Microsoft candidate used by Apple,Candidate used by both companies,Review tier
0,IntangibleAssetsNetExcludingGoodwillNoncurrent,AccountsReceivableNetCurrent,85.5,Goodwill,90.0,False,True,Review candidate


Strong text candidates: 0 of 1 Apple extensions. Each requires definition and context review.


## 5. Client recommendation to Apple management

Use the filing-specific tables above as the tag inventory and the candidate table as the action list. The memo below is generated from the observed Apple and Microsoft tag mix and names the exact Apple extension and standard-tag candidates identified in this analysis. Confirm each proposed replacement against the current US GAAP taxonomy definition, dimensional applicability, period type, units, and the underlying disclosure before changing the filing. If no suitable standard concept exists, retain a clearly defined extension rather than force a misleading match.

In [7]:
apple_extension_share = float(tag_mix.loc["Apple Inc.", "Extension share"])
microsoft_extension_share = float(tag_mix.loc["Microsoft Corporation", "Extension share"])
if apple_extension_share > microsoft_extension_share:
    relative_advice = (
        f"Apple's extension share ({apple_extension_share:.1%}) exceeds Microsoft's "
        f"({microsoft_extension_share:.1%}). Management should prioritize reviewing "
        "Apple extensions with high-scoring standard candidates and reduce extensions "
        "only where the standard concept is substantively equivalent."
    )
else:
    relative_advice = (
        f"Apple's extension share ({apple_extension_share:.1%}) is at or below Microsoft's "
        f"({microsoft_extension_share:.1%}). The observed mix does not support a blanket "
        "extension-reduction target; continue to prefer faithful standard concepts and "
        "retain extensions for genuinely company-specific disclosures."
    )

if candidate_comparison.empty:
    recommendation_rows = "No Apple extension tags were found in the selected filing; no replacement candidates are applicable."
else:
    memo_candidates = candidate_comparison[
        ["Extension tag", "Apple standard candidate", "Apple score", "Microsoft standard candidate", "Microsoft score"]
    ].copy()
    memo_candidates["Decision"] = memo_candidates["Apple score"].map(
        lambda score: "Review only; verify accounting equivalence" if score >= 80
        else "No replacement recommended from weak text match"
    )
    memo_candidates = memo_candidates.rename(
        columns={
            "Extension tag": "Apple extension to review",
            "Apple standard candidate": "Closest Apple standard tag",
            "Apple score": "Apple score / 100",
            "Microsoft standard candidate": "Closest Microsoft standard tag",
            "Microsoft score": "Microsoft score / 100",
        }
    )
    memo_headers = list(memo_candidates.columns)
    table_lines = [
        "| " + " | ".join(memo_headers) + " |",
        "| " + " | ".join("---" for _ in memo_headers) + " |",
    ]
    for values in memo_candidates.itertuples(index=False, name=None):
        cells = []
        for header, value in zip(memo_headers, values):
            text = f"{float(value):.1f}" if header.endswith("score / 100") else str(value)
            cells.append(text.replace("|", "\\|").replace("\n", " "))
        table_lines.append("| " + " | ".join(cells) + " |")
    recommendation_rows = "\n".join(table_lines)

memo = f"""### To: Apple Inc. Management  
**Subject: XBRL tag selection recommendations**

**Recommendation.** {relative_advice}

For each proposed pairing below, evaluate whether the standard concept captures the extension's exact definition and reported context. A text score is not a taxonomy interpretation. Where a pairing is confirmed, replace the extension with the listed standard concept in the relevant facts and document the mapping in the filing review process. Where it is not equivalent, keep the extension and improve its label, definition, and anchoring where applicable.

{recommendation_rows}

**Control actions:** (1) route strong text candidates to the reporting team for definition-level review; (2) confirm dimensional and period/unit compatibility before any replacement; (3) document the rationale for retained extensions; and (4) monitor the extension share as a diagnostic, not as a target that overrides faithful reporting. Microsoft matches are benchmark evidence only and do not by themselves establish that Apple should use a particular concept.

**Scope:** This screening covers distinct concepts present in the selected quarterly SEC dataset filings for Apple and Microsoft. Similarity rankings can miss semantically equivalent concepts and can suggest false positives; consult the authoritative taxonomy and the original filing before implementation."""
display(Markdown(memo))

### To: Apple Inc. Management  
**Subject: XBRL tag selection recommendations**

**Recommendation.** Apple's extension share (1.1%) is at or below Microsoft's (4.2%). The observed mix does not support a blanket extension-reduction target; continue to prefer faithful standard concepts and retain extensions for genuinely company-specific disclosures.

For each proposed pairing below, evaluate whether the standard concept captures the extension's exact definition and reported context. A text score is not a taxonomy interpretation. Where a pairing is confirmed, replace the extension with the listed standard concept in the relevant facts and document the mapping in the filing review process. Where it is not equivalent, keep the extension and improve its label, definition, and anchoring where applicable.

| Apple extension to review | Closest Apple standard tag | Apple score / 100 | Closest Microsoft standard tag | Microsoft score / 100 | Decision |
| --- | --- | --- | --- | --- | --- |
| IntangibleAssetsNetExcludingGoodwillNoncurrent | AccountsReceivableNetCurrent | 85.5 | Goodwill | 90.0 | Review only; verify accounting equivalence |

**Control actions:** (1) route strong text candidates to the reporting team for definition-level review; (2) confirm dimensional and period/unit compatibility before any replacement; (3) document the rationale for retained extensions; and (4) monitor the extension share as a diagnostic, not as a target that overrides faithful reporting. Microsoft matches are benchmark evidence only and do not by themselves establish that Apple should use a particular concept.

**Scope:** This screening covers distinct concepts present in the selected quarterly SEC dataset filings for Apple and Microsoft. Similarity rankings can miss semantically equivalent concepts and can suggest false positives; consult the authoritative taxonomy and the original filing before implementation.

## Sources and interpretation notes

- SEC [Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets.html), 2026 Q2 archive; the SEC cautions that these extracts may contain errors and are not a substitute for full filings.
- Apple and Microsoft filing metadata and facts are selected from `SUB`, `NUM`, `TAG`, and `PRE`; filing links can be constructed from each accession at `https://www.sec.gov/Archives/edgar/data/{CIK}/{accession-without-hyphens}/{accession}-index.htm`.
- Filing selection and financial-statement table merging use the repository helper in `xbrl_fs/statements.py`.
- `rapidfuzz` compares tag names only. Review complete definitions, taxonomy releases, facts, dimensions, units, and the filing's disclosure before adopting a recommendation.
- Counts are distinct tag names used in each selected filing; they are not counts of reported facts or a measure of filing quality.